# Volume Bar Aggregation Node

Streams `TradeTick` data from the Nautilus Parquet catalog through a `BacktestNode`,
aggregates it into volume bars inside the engine, and writes the bars back to the same
catalog under `data/bar/ETHUSDT-LINEAR.BYBIT-<SIZE>-VOLUME-LAST-EXTERNAL/`.

**Design**

- The node's `DataEngine` builds the bars with its `VolumeBarAggregator` when the collector
  actor subscribes to an `INTERNAL` volume bar type — tick-by-tick, the same event-driven
  code path used live.
- Ticks are streamed in chunks (`chunk_size`), so the full ~81 GB tick history never sits
  in memory at once.
- The collector re-labels finished bars as `EXTERNAL` and flushes them to the catalog in
  batches. `EXTERNAL` is what lets future backtests load them directly as pre-aggregated
  data (matching the existing 1-minute bar folder layout).
- Bars go into the **same catalog** rather than a new one: the instrument definition the
  bars depend on lives here, so a strategy backtest can source instrument + bars from a
  single `catalog_path`.

**Why volume bars** — sampling by traded volume rather than clock time yields returns
closer to IID Gaussian (Ané & Geman, 2000, *Journal of Finance*; Easley, López de Prado &
O'Hara, 2012, "The Volume Clock", *Journal of Portfolio Management*; López de Prado, 2018,
*Advances in Financial Machine Learning*, ch. 2). Note a fixed threshold means bar
frequency tracks activity: ETHUSDT volume grew substantially from 2020 to 2026, so
bars/day rises across the sample. Calibrate `VOLUME_PER_BAR` to the period you actually
train and trade on.

**Operational notes**

- A full 2020–2026 run processes ~81 GB of ticks — expect hours. Validate with a short
  `START`/`END` window first.
- Re-running appends duplicate bars, so the run cell refuses to start if the output bar
  folder already exists.


In [ ]:
%%writefile volume_bar_collector.py
"""Actor that collects internally aggregated volume bars and writes them to a catalog."""

from nautilus_trader.common.actor import Actor
from nautilus_trader.config import ActorConfig
from nautilus_trader.model.data import Bar, BarType
from nautilus_trader.model.enums import AggregationSource
from nautilus_trader.persistence.catalog import ParquetDataCatalog


class VolumeBarCollectorConfig(ActorConfig):
    """Configuration for VolumeBarCollector."""

    bar_type: str              # INTERNAL volume bar type to aggregate from ticks
    output_catalog_path: str   # Catalog directory to write the EXTERNAL bars into
    flush_threshold: int = 200_000  # Write to catalog every N bars to bound memory


class VolumeBarCollector(Actor):
    """Subscribes to internally aggregated volume bars and persists them as EXTERNAL bars."""

    def __init__(self, config: VolumeBarCollectorConfig) -> None:
        super().__init__(config)
        self._bar_type = BarType.from_str(config.bar_type)
        # Re-label EXTERNAL so future backtests can load them as pre-aggregated data
        self._external_type = BarType(
            self._bar_type.instrument_id,
            self._bar_type.spec,
            AggregationSource.EXTERNAL,
        )
        self._buffer: list[Bar] = []
        self.total_written = 0

    def on_start(self) -> None:
        self.subscribe_bars(self._bar_type)

    def on_bar(self, bar: Bar) -> None:
        self._buffer.append(
            Bar(
                self._external_type,
                bar.open,
                bar.high,
                bar.low,
                bar.close,
                bar.volume,
                bar.ts_event,
                bar.ts_init,
            )
        )
        if len(self._buffer) >= self.config.flush_threshold:
            self._flush()

    def on_stop(self) -> None:
        self._flush()
        self.log.info(f"Finished: {self.total_written} volume bars written to catalog")

    def _flush(self) -> None:
        if not self._buffer:
            return
        catalog = ParquetDataCatalog(self.config.output_catalog_path)
        catalog.write_data(self._buffer)
        self.total_written += len(self._buffer)
        self.log.info(f"Flushed {len(self._buffer)} volume bars (total {self.total_written})")
        self._buffer.clear()


In [ ]:
import sys
from pathlib import Path
from nautilus_trader.config import ImportableActorConfig
from centralNode import BacktestRunner

# Ensure the actor module written above is importable by the node
sys.path.insert(0, str(Path.cwd()))

# --- Parameters -------------------------------------------------------------
CATALOG_PATH = str((Path.cwd().parent / "nautilusDataCatalog").resolve())
INSTRUMENT_ID = "ETHUSDT-LINEAR.BYBIT"
VOLUME_PER_BAR = 1000          # Bar threshold in base units (ETH)
START = None                   # e.g. "2024-01-01"; None = start of tick data (2020-10-21)
END = None                     # e.g. "2024-02-01"; None = end of tick data (2026-06-28)
CHUNK_SIZE = 1_000_000         # Ticks streamed per chunk (bounds memory)
FLUSH_THRESHOLD = 200_000      # Bars buffered before each catalog write

BAR_TYPE_INTERNAL = f"{INSTRUMENT_ID}-{VOLUME_PER_BAR}-VOLUME-LAST-INTERNAL"
BAR_TYPE_EXTERNAL = f"{INSTRUMENT_ID}-{VOLUME_PER_BAR}-VOLUME-LAST-EXTERNAL"

actors = [
    ImportableActorConfig(
        actor_path="volume_bar_collector:VolumeBarCollector",
        config_path="volume_bar_collector:VolumeBarCollectorConfig",
        config={
            "bar_type": BAR_TYPE_INTERNAL,
            "output_catalog_path": CATALOG_PATH,
            "flush_threshold": FLUSH_THRESHOLD,
        },
    )
]

# --- Instantiate and run ----------------------------------------------------
node = BacktestRunner(
    actors=actors,
    catalog_path=CATALOG_PATH,
    instrument_id=INSTRUMENT_ID,
    start=START,
    end=END,
    chunk_size=CHUNK_SIZE,
)

node.run()

In [ ]:
# Guard: refuse to append into an existing bar folder (would create duplicate bars).
# To redo a run, delete the folder first: import shutil; shutil.rmtree(out_dir)
out_dir = Path(CATALOG_PATH) / "data" / "bar" / BAR_TYPE_EXTERNAL
if out_dir.exists():
    raise RuntimeError(f"{out_dir} already exists - delete it first to avoid duplicate bars")

results = node.run()
results[0]


In [ ]:
# Verify what was written: bar count, time coverage, and arrival rate
import pandas as pd
import pyarrow.dataset as ds

ts = ds.dataset(out_dir).to_table(columns=["ts_init"])["ts_init"].to_pandas()
first, last = pd.to_datetime(ts.min(), utc=True), pd.to_datetime(ts.max(), utc=True)
days = (last - first).total_seconds() / 86400

print(f"{len(ts):,} volume bars written to {out_dir.name}")
print(f"Coverage: {first} -> {last}")
print(f"Average bars/day: {len(ts) / days:.1f}")
assert ts.is_monotonic_increasing, "Bar timestamps are not sorted"
